# Q1 | Neural spike patterns

Compare regular, irregular, and bursting spike trains over a five-second recording.

## A. Models and assumptions

Describe the generation rules, parameters, randomness, and recording boundaries.

In [ ]:
duration_s = 5.0

### Regular firing

The first spike occurs at 0.05 seconds. Each subsequent spike occurs 0.10 seconds (100 ms) after the previous one. The interval gives a baseline of 10 spikes per second for comparing the three models. The start time places the first spike inside the recording window.

The model is deterministic. The positive interval keeps the spike times strictly increasing. The recording includes times from zero up to, but excluding, five seconds.

In [ ]:
import numpy as np

regular_start_s = 0.05
regular_interval_s = 0.10
regular_spike_times_s = np.arange(regular_start_s, duration_s, regular_interval_s)

print(f"Number of regular spikes: {regular_spike_times_s.size}")
print(f"First five spike times (s): {regular_spike_times_s[:5]}")
print(f"Last spike time (s): {regular_spike_times_s[-1]:.2f}")

### Irregular firing

Real neurons cannot fire again immediately after a spike. We assume a fixed absolute refractory period of 2 ms (0.002 seconds), a rounded value at the upper end of the commonly cited range of about 1–2 ms.

Each waiting time is the 0.002 second refractory period plus an independent draw from an exponential distribution with a mean of 0.098 seconds. The refractory part is deterministic and the exponential part is stochastic. Short waits are more common than long ones, but none is shorter than 2 ms. The mean wait is 0.002 + 0.098 = 0.10 seconds, giving the same expected rate of 10 spikes per second as the regular model. The first waiting time, measured from zero, follows the same rule, as though a spike had just occurred.

Add each positive wait to the previous time and keep spike times below the recording duration. A fixed seed lets us reproduce one example. Changing the seed gives another recording from the same model.

The regular model (100 ms gaps) and the bursting model (gaps of at least 10 ms) already satisfy this limit.

Limitation: refractory periods vary between neuron types, and the relative refractory period, when firing is possible but harder, is not modeled.

In [ ]:
def simulate_irregular(duration_s, mean_gap_s, refractory_s, seed):
    if not np.isfinite(mean_gap_s) or mean_gap_s <= 0:
        raise ValueError("The mean gap must be finite and positive.")
    if not np.isfinite(duration_s) or duration_s < 0:
        raise ValueError("The duration must be finite and nonnegative.")

    if not np.isfinite(refractory_s) or not 0 <= refractory_s < mean_gap_s:
        raise ValueError("The refractory period must be nonnegative and shorter than the mean gap.")

    rng = np.random.default_rng(seed)
    spike_times = []
    time_s = 0.0
    while time_s < duration_s:
        # Fixed refractory period plus a random wait; the total still averages mean_gap_s.
        gap_s = refractory_s + rng.exponential(mean_gap_s - refractory_s)
        if gap_s == 0:  # Only possible if the refractory period is set to zero.
            continue
        time_s += gap_s
        if time_s < duration_s:
            spike_times.append(time_s)
    return np.array(spike_times)


irregular_mean_gap_s = regular_interval_s
refractory_s = 0.002
irregular_spike_times_s = simulate_irregular(
    duration_s, irregular_mean_gap_s, refractory_s, seed=42,
)
print(f"Number of irregular spikes: {irregular_spike_times_s.size}")
print("First five spike times (s):", irregular_spike_times_s[:5].round(3))
print("First four gaps (s):", np.diff(irregular_spike_times_s[:5]).round(3))

### Bursting firing

Spikes arrive in bursts of five, with 0.01 seconds (10 ms) between spikes inside a burst. Between bursts the neuron is quiet. The quiet gap runs from the last spike of one burst to the first spike of the next. Each gap is 0.10 seconds plus an independent exponential wait with a mean of 0.36 seconds, so the mean gap is 0.46 seconds. A burst lasts 0.04 seconds, so a new burst begins every 0.5 seconds on average. Five spikes per 0.5 seconds gives the same expected rate of 10 spikes per second as the other two models.

The number of spikes per burst, the spacing inside a burst, and the minimum quiet gap are deterministic. The exponential part of each quiet gap is stochastic. The minimum gap is ten times the spacing inside a burst, so neighboring bursts never merge and the two kinds of interval stay distinct.

The recording starts as though a burst had just ended, so a quiet gap comes before the first burst. Spikes at or after five seconds are dropped. The final burst can therefore be partial, and it is the only burst allowed to have fewer than five spikes. Each burst starts after the previous spike and its spikes are placed in increasing order, so the spike times stay chronological.

Limitation: real bursts vary in size and internal spacing. This model holds both fixed so that the only source of variability is the timing of the bursts.

In [ ]:
def simulate_bursting(duration_s, spikes_per_burst, within_burst_gap_s,
                      min_quiet_s, mean_extra_quiet_s, seed):
    if not np.isfinite(duration_s) or duration_s < 0:
        raise ValueError("The duration must be finite and nonnegative.")
    if int(spikes_per_burst) != spikes_per_burst or spikes_per_burst < 1:
        raise ValueError("Each burst must contain a positive whole number of spikes.")
    if not np.isfinite(within_burst_gap_s) or within_burst_gap_s <= 0:
        raise ValueError("The gap inside a burst must be finite and positive.")
    if not np.isfinite(min_quiet_s) or min_quiet_s <= within_burst_gap_s:
        raise ValueError("The minimum quiet gap must be longer than the gap inside a burst.")
    if not np.isfinite(mean_extra_quiet_s) or mean_extra_quiet_s < 0:
        raise ValueError("The mean extra quiet time must be finite and nonnegative.")

    rng = np.random.default_rng(seed)
    burst_offsets_s = within_burst_gap_s * np.arange(spikes_per_burst)
    spike_times = []
    last_spike_s = 0.0  # The recording starts as though a burst had just ended.
    while True:
        quiet_gap_s = min_quiet_s + rng.exponential(mean_extra_quiet_s)
        burst_start_s = last_spike_s + quiet_gap_s
        if burst_start_s >= duration_s:
            break
        burst_times_s = burst_start_s + burst_offsets_s
        burst_times_s = burst_times_s[burst_times_s < duration_s]  # A final burst can be partial.
        spike_times.extend(burst_times_s)
        last_spike_s = burst_times_s[-1]
    return np.array(spike_times)


spikes_per_burst = 5
within_burst_gap_s = 0.01
min_quiet_s = 0.10
mean_extra_quiet_s = 0.36

bursting_spike_times_s = simulate_bursting(
    duration_s, spikes_per_burst, within_burst_gap_s,
    min_quiet_s, mean_extra_quiet_s, seed=42,
)
print(f"Number of bursting spikes: {bursting_spike_times_s.size}")
print("First ten spike times (s):", bursting_spike_times_s[:10].round(3))
print("First nine gaps (s):", np.diff(bursting_spike_times_s[:10]).round(3))

## B. Compare firing patterns

Use raster plots, inter-spike intervals, and numerical summaries to compare the three models.

## C. Validate the simulation

Check temporal behavior, a statistical property, and an edge case.

## D. Compare new realizations

Repeat the stochastic simulations and explain how to identify the firing patterns from observations.